# Phase 5t: Doublon Geometric SWAP Gate — Technical Notebook

**Target:** Kiefer et al. (Nature 2026, arXiv:2507.22112) — formally verified in Lean 4.

This notebook reproduces every Lean-verified result of Phase 5t end-to-end by importing from `src.experimental.doublon_gate` (which wraps `src.fermi_hubbard.dimer`). No new physics — everything is a cross-check of the Lean formalization against numpy `eigh`/`linalg`.

**Structure:**
1. Hamiltonian definitions (W2)
2. Singlet sector + dark state at U=0 (W4)
3. BDI chiral symmetry (W5)
4. Geometric SWAP operator (W6C)
5. 6×6 block-diagonal lift (W6-deferred)
6. Direct-exchange vs superexchange scaling (W7)
7. Minimal Berry-phase theorem (W8 Target A)

<!-- viz-ref: fig_doublon_gate_spectrum -->

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.fermi_hubbard.dimer import (
    H_singlet, H_full,
    chiral_op, chiral_proj_plus, chiral_proj_minus,
    dark_vec, bright_vec_plus, bright_vec_minus,
    dark_state_theta_norm,
    u_swap_singlet, u_swap_adapted, u_swap_action_on_kernel,
    E_plus, E_minus, J_superexchange, J_leading_superexchange,
    E_plus_char_residual, E_minus_char_residual,
    E_plus_eigenvector, E_minus_eigenvector, antisymmetric_doublon_vec,
    geometric_phase_loop_check,
)
from src.experimental.doublon_gate import (
    exact_diagonalize,
    dimer_spectrum_at_U0,
    swap_action_on_singlet,
    scaling_comparison_curves,
    bench_superexchange_bound,
    gate_6x6_unitarity_witness,
)
from src.core.visualizations import COLORS

np.set_printoptions(precision=4, suppress=True)

## 1. Hamiltonian definitions (W2)

The 3×3 singlet-sector Hamiltonian in the `{|D+⟩, |D-⟩, |s⟩}` basis and the 6×6 full Hamiltonian. Every matrix entry is Lean-verified (Section 1 of `FermiHubbardDimer.lean`, T1-T7).

In [ ]:
t, delta, U = 1.0, 0.6, 2.0
H3 = H_singlet(t, delta, U)
H6 = H_full(t, delta, U)
print('H_singlet (3×3):')
print(H3)
print(f'\ntrace = {np.trace(H3)} (Lean T7: 2U = {2*U})')
print(f'det   = {np.linalg.det(H3):.4f} (Lean T6: -4U·t² = {-4*U*t**2})')

## 2. Dark state at U = 0 (W4)

At U=0 the spectrum is exactly `{0, +gap, -gap}` where `gap = √(Δ²+4t²)` (Lean W4p). The zero eigenvector is `darkVec = (0, 2t, Δ)` (Lean W2 T2).

In [ ]:
t, delta = 1.0, 0.6
gap = np.sqrt(delta**2 + 4*t**2)
spec = dimer_spectrum_at_U0(t, delta)
print(f'Spectrum: {spec}  (expected: {sorted([0.0, gap, -gap])})')
print(f'Gap:      {gap:.4f}')

dv = dark_vec(t, delta)
H0 = H_singlet(t, delta, 0.0)
print(f'H·darkVec = {H0 @ dv}  (expected: [0, 0, 0])')

## 3. BDI chiral symmetry (W5)

The chiral operator Γ = diag(+1, -1, -1) anticommutes with H_singlet(t, Δ, 0). The dark state lives in the -1 eigenspace of Γ (Lean W5e). The projectors P± = (1±Γ)/2 are orthogonal, complete, and idempotent (Lean W5l-W5o).

In [ ]:
Gamma = chiral_op()
anticomm = Gamma @ H0 + H0 @ Gamma
print(f'||ΓH + HΓ||_F = {np.linalg.norm(anticomm):.2e}  (Lean W5a: exactly zero)')

print(f'Γ·darkVec = {Gamma @ dv}  (Lean W5e: equals -darkVec = {-dv})')

Pp, Pm = chiral_proj_plus(), chiral_proj_minus()
print(f'\nP+ + P- =\n{Pp + Pm}  (Lean W5o: identity)')
print(f'P+ · P- = {np.allclose(Pp @ Pm, 0)}  (Lean W5n: orthogonal)')

## 4. Geometric SWAP operator (W6C)

`U_SWAP_singlet` is the Householder reflection `I - (2/gap²)·darkVec⊗darkVec`. It flips the sign of the dark state (W6C-A1), acts as identity on both bright states (W6C-A2/A3), and is unitary (W6C-S3 and W6C-U1).

In [ ]:
witness = swap_action_on_singlet(t, delta)
U3 = witness['U_SWAP']
print(f'U_SWAP =\n{U3}')
print(f'\nU · darkVec     = {witness["U_times_dark"]}  (expected: -darkVec = {-witness["dark"]})')
print(f'U · brightVec+ = {witness["U_times_bright_plus"]}')
print(f'\nU·Uᵀ - I Frobenius norm: {np.linalg.norm(U3 @ U3.T - np.eye(3)):.2e}')
print(f'det U = {np.linalg.det(U3):.4f}  (Lean W6C-D1: -1)')
print(f'tr U  = {np.trace(U3):.4f}  (Lean W6C-T1: +1)')
print(f'eigs  = {sorted(np.linalg.eigvalsh(U3))}  (Lean W6C-E1: {{-1,+1,+1}})')

## 5. 6×6 block-diagonal lift (W6-deferred)

`U_SWAP_adapted = fromBlocks U_SWAP_singlet 0 0 I` — the singlet-sector SWAP acts as W6C says, the triplet block is identity. Lean W6D-S3 gives the 6×6 unitarity; W6D-A1/A2 give the block-action theorems.

In [ ]:
U6 = u_swap_adapted(t, delta)
wit = gate_6x6_unitarity_witness(t, delta)
print('6×6 unitarity witnesses (all should be ≈ 0):')
for k, v in wit.items():
    print(f'  {k:25s}: {v:.2e}')

eigs = sorted(np.linalg.eigvalsh(U6))
print(f'\n6×6 eigenvalues: {eigs}  (expected: {{-1,+1,+1,+1,+1,+1}})')

## 6. Direct-exchange vs superexchange scaling (W7)

`E_plus(t, U) = (U + √(U² + 16·t²))/2` is the upper eigenvalue of the 2×2 symmetric block at Δ=0. Direct exchange: `E_plus(t, 0) = 2|t|`, Taylor coefficient at U=0 is exactly 1/2 (Lean W7d). Superexchange: `J → 4t²/U` with bound `|J − 4t²/U| ≤ 16t⁴/U³` for `U ≥ 4|t|` (Lean W7i).

In [ ]:
t = 1.0
Us = np.linspace(0.0, 10.0, 200)
curves = scaling_comparison_curves(t, Us)

factors = np.linspace(1.0, 50.0, 100)
bench = bench_superexchange_bound(t, factors)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Direct exchange: E_+(t,0)=2|t|, slope 1/2 at U=0',
        'Superexchange bound |J − 4t²/U| ≤ 16t⁴/U³',
    ),
    horizontal_spacing=0.12,
)
fig.add_trace(go.Scatter(x=Us, y=curves['E_plus'],
                         name='E+(t, U)', line=dict(color=COLORS['steel_blue'], width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=Us, y=curves['E_minus'],
                         name='E-(t, U)', line=dict(color=COLORS['dissipative'], width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=Us, y=curves['direct_linear'],
                         name='2|t|+U/2 (Taylor)',
                         line=dict(color=COLORS['cross'], width=1.2, dash='dash')),
              row=1, col=1)

fig.add_trace(go.Scatter(x=bench['U'], y=bench['residual'],
                         name='|J − 4t²/U| actual',
                         line=dict(color=COLORS['steel_blue'], width=2)),
              row=1, col=2)
fig.add_trace(go.Scatter(x=bench['U'], y=bench['bound'],
                         name='Lean W7i: 16t⁴/U³',
                         line=dict(color=COLORS['dissipative'], width=2, dash='dash')),
              row=1, col=2)

fig.update_xaxes(title_text='U', row=1, col=1)
fig.update_yaxes(title_text='Eigenvalue', row=1, col=1)
fig.update_xaxes(title_text='U', type='log', row=1, col=2)
fig.update_yaxes(title_text='|residual|', type='log', row=1, col=2)
fig.update_layout(height=380, width=900, showlegend=True,
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Cross-check: characteristic equations (W7j, W7k)
for t, U in [(1.0, 0.0), (0.7, 2.5), (1.5, -1.0), (2.0, 10.0)]:
    r_plus = E_plus_char_residual(t, U)
    r_minus = E_minus_char_residual(t, U)
    print(f't={t:>5}, U={U:>6}: E_plus char residual {r_plus:+.2e}, E_minus {r_minus:+.2e}')

## 7. Minimal Berry-phase theorem (W8 Target A)

The θ-parameterized dark state `|dark(θ)⟩ = (0, sin θ, cos θ)` picks up −1 after a π-sweep (Lean W8a). Along the kernel-angle path `Δ sin θ = 2t cos θ`, the dynamical phase vanishes identically (Lean W8d). Together: the accumulated phase is purely geometric and equal to −1 (Lean W8e).

This is the minimal finite-dimensional algebraic core of the Kiefer et al. Berry-phase mechanism. Target B (full adiabatic holonomy + Berry connection) is deferred to future work.

In [ ]:
# Verify the sign-flip on a grid of θ values
thetas = np.linspace(-np.pi, np.pi, 20)
errs = []
for th in thetas:
    w = geometric_phase_loop_check(th)
    errs.append((w['sign_flip_err'], w['periodicity_err'], w['unit_norm_err']))
errs = np.array(errs)
print(f'Max sign-flip err over 20 θ:   {errs[:,0].max():.2e}  (Lean W8a exact)')
print(f'Max periodicity err:           {errs[:,1].max():.2e}  (Lean W8b exact)')
print(f'Max unit-norm err:             {errs[:,2].max():.2e}  (Lean W8c exact)')

# Dynamical phase vanishes along kernel-angle path
thetas_path = np.linspace(0.1, np.pi/2, 10)
max_exp = 0.0
for th in thetas_path:
    t_th = np.sin(th) / 2.0
    delta_th = np.cos(th)
    H = H_singlet(t_th, delta_th, 0.0)
    v = dark_state_theta_norm(th)
    max_exp = max(max_exp, abs(v @ (H @ v)))
print(f'\nMax dynamical-phase expectation |⟨dark|H|dark⟩|: {max_exp:.2e}  (Lean W8d exact)')

In [ ]:
# Visualize the sign flip as θ sweeps through π
thetas = np.linspace(0, 2*np.pi, 200)
s1 = np.array([dark_state_theta_norm(th)[1] for th in thetas])  # sin component
s2 = np.array([dark_state_theta_norm(th)[2] for th in thetas])  # cos component

fig = go.Figure()
fig.add_trace(go.Scatter(x=thetas, y=s1,
                         name='⟨D-|dark(θ)⟩ = sin(θ)',
                         line=dict(color=COLORS['steel_blue'], width=2)))
fig.add_trace(go.Scatter(x=thetas, y=s2,
                         name='⟨s|dark(θ)⟩ = cos(θ)',
                         line=dict(color=COLORS['dissipative'], width=2)))
fig.add_vline(x=np.pi, line_dash='dash', line_color='red', opacity=0.6,
              annotation_text='θ = π: sign flip')
fig.update_layout(
    title='W8a: |dark(θ+π)⟩ = −|dark(θ)⟩ (both components flip sign)',
    xaxis_title='θ', yaxis_title='Dark-state component',
    xaxis=dict(tickvals=[0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi],
               ticktext=['0', 'π/2', 'π', '3π/2', '2π']),
    height=400, width=820, plot_bgcolor='white', paper_bgcolor='white',
)
fig.show()

## Summary — what's formally verified

The full Phase 5t formalization lives in `lean/SKEFTHawking/FermiHubbardDimer.lean`:

| Section | Wave | Theorems | Focus |
|---------|------|----------|-------|
| 1–4 | W2 Layer-1 | 15 | Hamiltonian, dark state, T1-T10c |
| 5 | W3 block-match | 10 | symmetry-adapted matrix actions |
| 6–9 | W4 U=0 spectrum | 20 | charpoly, brights, full spectrum |
| 10–11 | W5 BDI symmetry | 21 | ΓHΓ=−H, chiral projectors, zero-mode unique |
| 12–14 | W6A/B/C SWAP | 39 | EuclideanSpace, OrthonormalBasis, Householder |
| 15, 18 | W6 round-2 + deferred | 12 | scalar actions + 6×6 lift |
| 16–17 | W7 scaling + round-2 | 17 | Vieta, Lipschitz, superexch. bound |
| 19 | W8 Berry phase | 5 | sign flip, vanishing dynamical phase |
| **Total** | **W2–W9** | **139** | **zero sorry, zero new axioms** |

See `papers/paper18_doublon_gate/paper_draft.tex` for the full writeup.